In [1]:
import pandas as pd
import numpy as np

In [2]:
class DataLoader:
    """Handles loading all CSV datasets used in the project."""
    @classmethod
    def load_training_data(self):
        """Loads 2023 / training data."""
        solcast = pd.read_csv("Data/Solcast/Solcast_2023.csv", parse_dates=['DateTime'], index_col='DateTime')
        deop = pd.read_csv("Data/DEOP/2023_DEOP_Interp.csv", parse_dates=['DateTime'], index_col='DateTime')
        expected = pd.read_csv("Data/DEOP/2023_Expected.csv", parse_dates=['DateTime'], index_col='DateTime')
        return solcast, deop, expected
    @classmethod
    def load_testing_data(self):
        """Loads 2022 / test data."""
        deop_2022 = pd.read_csv("Data/DEOP/2022_DEOP_Interp.csv", parse_dates=['DateTime'], index_col='DateTime')
        solcast_2022 = pd.read_csv("Data/Solcast/Solcast_2022.csv", parse_dates=['DateTime'], index_col='DateTime')
        expected_2022 = pd.read_csv("Data/DEOP/2022_Expected.csv", parse_dates=['DateTime'], index_col='DateTime')

        return deop_2022, solcast_2022, expected_2022

In [3]:
class SolarReconstructor:
    """Reconstructs missing solar data."""
    @staticmethod
    def get_problem_dates(expected, actual, threshold):
        """Identifies days where solar generation is below the threshold."""
        daily_actual = actual.resample('D').sum()
        daily_expected = expected.resample('D').sum()
        daily_ratio = daily_actual['power-gen-pv-ave'] / daily_expected['expected']
        
        return daily_ratio[daily_ratio < threshold].index.date

    @staticmethod
    def calculate_performance_ratios(expected_clean, actual_clean):
        """Generates seasonal and daily performance ratio maps."""
        merged = expected_clean.merge(actual_clean, left_index=True, right_index=True)
        merged['ratio'] = (merged['power-gen-pv-ave'] / merged['expected']).replace([np.inf, -np.inf], np.nan).fillna(0)

        monthly_ratios = merged.groupby([merged.index.month, merged.index.time])['ratio'].mean()

        overall_ratios = merged.groupby(merged.index.time)['ratio'].mean()
        
        return monthly_ratios, overall_ratios

    @classmethod
    def reconstruct_data(cls, actual, expected, threshold=0.01):
        """Reconstructs the missing data points."""
        prob_dates = cls.get_problem_dates(expected, actual, threshold)
        is_prob = pd.Series(actual.index.date).isin(prob_dates).values
        
        expected_clean, actual_clean = expected[~is_prob], actual[~is_prob]
        expected_problem = expected[is_prob]

        monthly_ratios, overall_ratios = cls.calculate_performance_ratios(expected_clean, actual_clean)

        def calculate_repair(row):
            m, t = row.name.month, row.name.time()
            ratio = monthly_ratios.get((m, t), overall_ratios.get(t, 0.85))
            return row['expected'] * ratio

        imputed_series = expected_problem.apply(calculate_repair, axis=1)
        imputed_df = pd.DataFrame(imputed_series, columns=['power-gen-pv-ave'])

        return pd.concat([actual_clean, imputed_df]).sort_index()